# Air Quality ML Project: Notebook 07
## Final Model Selection, Export and Inference Demonstration

Notebooks 02–06 trained and compared models, but every fitted estimator lived only in
memory. This notebook does the last step a real project needs: it **retrains the
selected models, persists them to disk, reloads them, and demonstrates prediction on
unseen data**.

**Why this matters for the assignment.** The Model Validation criterion asks for work
"leading to select the most suitable model at last", and the overall marking criteria
reference a demonstration. A saved model turns "Random Forest scored 0.753" into
something one can actually run and show.

**Models exported here**

| Task | Selected model | Justification |
|---|---|---|
| Multiclass AQI category | Random Forest, `class_weight='balanced'` | Best macro-F1 (0.753) and best minority recall at no extra cost |
| Binary unhealthy hour | Random Forest (tuned) | Best F1 (0.921), ROC-AUC 0.977 |
| PM2.5 regression | Random Forest | R² 0.895, within 0.003 of the MLP but far more interpretable and it exposes feature importances |

> **A note on the regression choice.** The MLP scored marginally higher (R² 0.898 against
> 0.895), but Random Forest is selected for deployment. The difference is well within
> run-to-run variation, while Random Forest trains faster, is deterministic given a seed,
> and provides feature importances that support the analysis. Choosing the marginally
> lower-scoring model for defensible engineering reasons, and saying so, is better
> practice than always shipping the top row of the table.

> Run **Notebook 01** first, this loads `beijing_aqi_clean.parquet`.


## 0. Setup

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
import time

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             mean_squared_error, mean_absolute_error, r2_score)

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)
print("Setup complete. Models will be saved to:", MODEL_DIR + "/")

Setup complete. Models will be saved to: models/


## 1. Load Data and Rebuild the Shared Components

These definitions match Notebooks 02–04 exactly, including `random_state=42` and the
same 30,000-row stratified sample, so the metrics reported below reproduce the figures
quoted in the report rather than drifting from them.

In [2]:
df = pd.read_parquet("beijing_aqi_clean.parquet")

NUMERIC_FEATURES = ["PM10","SO2","NO2","CO","O3","TEMP","PRES","DEWP","RAIN","WSPM",
                    "hour_sin","hour_cos","month_sin","month_cos"]
CATEGORICAL_FEATURES = ["wd","station","season"]
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

def build_preprocessor():
    numeric = Pipeline([("impute", SimpleImputer(strategy="median")),
                        ("scale",  StandardScaler())])
    categorical = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                            ("onehot", OneHotEncoder(handle_unknown="ignore",
                                                     sparse_output=False))])
    return ColumnTransformer([("num", numeric, NUMERIC_FEATURES),
                              ("cat", categorical, CATEGORICAL_FEATURES)])

print("Rows available:", f"{len(df):,}")
print("Features used:", len(ALL_FEATURES), "( PM2.5 excluded from classification inputs )")

Rows available: 420,768
Features used: 17 ( PM2.5 excluded from classification inputs )


## 2. Model 1: Multiclass AQI Category Classifier

Random Forest with balanced class weights. **PM2.5 is excluded from the features**, as
established in Notebook 02, the target is derived from it, so including it would be
target leakage.

In [3]:
clf_data = df.dropna(subset=["AQI_Category"]).copy()
_, sample = train_test_split(clf_data, test_size=30_000/len(clf_data),
                             stratify=clf_data["AQI_Category"], random_state=RANDOM_STATE)

Xc, yc = sample[ALL_FEATURES], sample["AQI_Category"]
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(
    Xc, yc, test_size=0.2, stratify=yc, random_state=RANDOM_STATE)

model_multiclass = Pipeline([
    ("pre", build_preprocessor()),
    ("model", RandomForestClassifier(n_estimators=100, class_weight="balanced",
                                     n_jobs=-1, random_state=RANDOM_STATE)),
])

t0 = time.time(); model_multiclass.fit(Xc_tr, yc_tr); print(f"Trained in {time.time()-t0:.1f}s")
pred = model_multiclass.predict(Xc_te)
print(f"Accuracy : {accuracy_score(yc_te, pred):.3f}")
print(f"Macro-F1 : {f1_score(yc_te, pred, average='macro'):.3f}")

Trained in 0.3s
Accuracy : 0.755
Macro-F1 : 0.753


## 3. Model 2: Binary Unhealthy-Hour Classifier

The tuned Random Forest configuration selected in Notebook 03.

In [4]:
bin_data = df.dropna(subset=["Unhealthy"]).copy()
bin_data["Unhealthy"] = bin_data["Unhealthy"].astype(int)
_, sample_b = train_test_split(bin_data, test_size=30_000/len(bin_data),
                               stratify=bin_data["Unhealthy"], random_state=RANDOM_STATE)

Xb, yb = sample_b[ALL_FEATURES], sample_b["Unhealthy"]
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(
    Xb, yb, test_size=0.2, stratify=yb, random_state=RANDOM_STATE)

model_binary = Pipeline([
    ("pre", build_preprocessor()),
    # Exact configuration selected by RandomizedSearchCV in Notebook 03
    ("model", RandomForestClassifier(n_estimators=200, max_depth=20,
                                     min_samples_leaf=1, max_features="sqrt",
                                     n_jobs=-1, random_state=RANDOM_STATE)),
])

t0 = time.time(); model_binary.fit(Xb_tr, yb_tr); print(f"Trained in {time.time()-t0:.1f}s")
pred_b  = model_binary.predict(Xb_te)
proba_b = model_binary.predict_proba(Xb_te)[:, 1]
print(f"Accuracy : {accuracy_score(yb_te, pred_b):.3f}")
print(f"F1       : {f1_score(yb_te, pred_b):.3f}")
print(f"ROC-AUC  : {roc_auc_score(yb_te, proba_b):.3f}")

Trained in 0.3s
Accuracy : 0.921
F1       : 0.921
ROC-AUC  : 0.977


## 4. Model 3: PM2.5 Regression Model

Here PM2.5 is the target, so no exclusion applies.

In [5]:
reg_data = df.dropna(subset=["PM2.5"]).copy()
sample_r = reg_data.sample(n=20_000, random_state=RANDOM_STATE)

Xr, yr = sample_r[ALL_FEATURES], sample_r["PM2.5"]
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(
    Xr, yr, test_size=0.2, random_state=RANDOM_STATE)

model_regression = Pipeline([
    ("pre", build_preprocessor()),
    ("model", RandomForestRegressor(n_estimators=100, n_jobs=-1,
                                    random_state=RANDOM_STATE)),
])

t0 = time.time(); model_regression.fit(Xr_tr, yr_tr); print(f"Trained in {time.time()-t0:.1f}s")
pred_r = model_regression.predict(Xr_te)
print(f"RMSE : {np.sqrt(mean_squared_error(yr_te, pred_r)):.2f} ug/m3")
print(f"MAE  : {mean_absolute_error(yr_te, pred_r):.2f} ug/m3")
print(f"R2   : {r2_score(yr_te, pred_r):.3f}")

Trained in 0.7s
RMSE : 26.20 ug/m3
MAE  : 14.45 ug/m3
R2   : 0.895


## 5. Export the Models

`joblib` is used rather than `pickle` because it handles large NumPy arrays more
efficiently. Each saved object is the **entire pipeline**, not just the estimator, so
imputation, scaling and one-hot encoding travel with the model. A reloaded model
therefore accepts raw, unprocessed input and handles preparation internally, which
removes the most common source of train/serve mismatch.

Compression is enabled because Random Forest stores every tree.

In [6]:
artifacts = {
    "model_aqi_multiclass.joblib": model_multiclass,
    "model_unhealthy_binary.joblib": model_binary,
    "model_pm25_regression.joblib": model_regression,
}

for fname, obj in artifacts.items():
    path = os.path.join(MODEL_DIR, fname)
    joblib.dump(obj, path, compress=3)
    print(f"{fname:34s} {os.path.getsize(path)/1e6:6.1f} MB")

model_aqi_multiclass.joblib          18.2 MB
model_unhealthy_binary.joblib        11.6 MB
model_pm25_regression.joblib         23.1 MB


### Save the metadata alongside the models
A model file without its context is close to useless six months later. This records the
feature order, the class labels, the decision threshold and the training provenance.

In [7]:
metadata = {
    "project": "Beijing Multi-Site Air Quality - CT046-3-M-AML",
    "dataset": "UCI ML Repository, Dataset ID 501",
    "trained_on": time.strftime("%Y-%m-%d"),
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "feature_order": ALL_FEATURES,
    "class_labels": list(yc.cat.categories),
    "pm25_excluded_from_classification": True,
    "recommended_binary_threshold": 0.25,
    "threshold_rationale": "Selected for >=98% recall on unhealthy hours; see Notebook 03.",
    "training_sample_sizes": {"multiclass": 30000, "binary": 30000, "regression": 20000},
    "random_state": RANDOM_STATE,
}

joblib.dump(metadata, os.path.join(MODEL_DIR, "model_metadata.joblib"))
for k, v in metadata.items():
    if not isinstance(v, (list, dict)):
        print(f"{k:38s} {v}")

project                                Beijing Multi-Site Air Quality - CT046-3-M-AML
dataset                                UCI ML Repository, Dataset ID 501
trained_on                             2026-08-07
pm25_excluded_from_classification      True
recommended_binary_threshold           0.25
threshold_rationale                    Selected for >=98% recall on unhealthy hours; see Notebook 03.
random_state                           42


## 6. Reload and Verify

Saving a model is only half the job, the artifact must be shown to work after
reloading. Here we reload from disk in a fresh object and confirm the predictions match
those of the in-memory model exactly.

In [8]:
loaded_multiclass = joblib.load(os.path.join(MODEL_DIR, "model_aqi_multiclass.joblib"))
loaded_binary     = joblib.load(os.path.join(MODEL_DIR, "model_unhealthy_binary.joblib"))
loaded_regression = joblib.load(os.path.join(MODEL_DIR, "model_pm25_regression.joblib"))
loaded_meta       = joblib.load(os.path.join(MODEL_DIR, "model_metadata.joblib"))

same_mc = (loaded_multiclass.predict(Xc_te) == pred).all()
same_bin = (loaded_binary.predict(Xb_te) == pred_b).all()
same_reg = np.allclose(loaded_regression.predict(Xr_te), pred_r)

print("Reloaded multiclass predictions identical:", same_mc)
print("Reloaded binary predictions identical    :", same_bin)
print("Reloaded regression predictions identical:", same_reg)
assert same_mc and same_bin and same_reg, "Reloaded model does not reproduce predictions!"
print()
print("All three models reload correctly and reproduce their predictions exactly.")

Reloaded multiclass predictions identical: True
Reloaded binary predictions identical    : True
Reloaded regression predictions identical: True

All three models reload correctly and reproduce their predictions exactly.


## 7. Inference Demonstration: Predicting for New Readings

This is what one would show in a demonstration. The function accepts a **raw sensor
reading**, the kind of record a monitoring station produces, and returns all three
predictions. Note that the caller supplies no PM2.5 value for the classification
outputs; that is the whole point.

In [9]:
def predict_air_quality(reading: dict, threshold: float = 0.25) -> dict:
    """Predict AQI category, unhealthy-hour flag and PM2.5 from a raw sensor reading.

    `reading` should contain the pollutant, weather and time fields listed in
    metadata['feature_order']. Missing values are handled by the pipeline.
    """
    row = pd.DataFrame([reading])

    # Derive the cyclical time features if the caller supplied plain hour/month
    if "hour" in reading and "hour_sin" not in reading:
        row["hour_sin"] = np.sin(2*np.pi*row["hour"]/24)
        row["hour_cos"] = np.cos(2*np.pi*row["hour"]/24)
    if "month" in reading and "month_sin" not in reading:
        row["month_sin"] = np.sin(2*np.pi*row["month"]/12)
        row["month_cos"] = np.cos(2*np.pi*row["month"]/12)
        season_map = {12:"Winter",1:"Winter",2:"Winter", 3:"Spring",4:"Spring",5:"Spring",
                      6:"Summer",7:"Summer",8:"Summer", 9:"Autumn",10:"Autumn",11:"Autumn"}
        row["season"] = row["month"].map(season_map)

    for col in ALL_FEATURES:                 # fill anything still absent
        if col not in row.columns:
            row[col] = np.nan
    row = row[ALL_FEATURES]

    category   = loaded_multiclass.predict(row)[0]
    risk       = float(loaded_binary.predict_proba(row)[0, 1])
    prediction = float(loaded_regression.predict(row)[0])

    return {
        "predicted_AQI_category": str(category),
        "unhealthy_probability":  round(risk, 3),
        "unhealthy_flag":         bool(risk >= threshold),
        "predicted_PM25":         round(prediction, 1),
    }

print("Inference function ready.")

Inference function ready.


### Example 1: a severe winter smog hour
High co-pollutants, cold, almost no wind.

In [10]:
winter_smog = {
    "PM10": 260.0, "SO2": 45.0, "NO2": 105.0, "CO": 3400.0, "O3": 18.0,
    "TEMP": 3.5, "PRES": 1017.0, "DEWP": -3.0, "RAIN": 0.0, "WSPM": 1.0,
    "wd": "NE", "station": "Aotizhongxin", "hour": 20, "month": 1,
}
result = predict_air_quality(winter_smog)
for k, v in result.items():
    print(f"{k:26s} {v}")

predicted_AQI_category     Hazardous
unhealthy_probability      1.0
unhealthy_flag             True
predicted_PM25             239.9


### Example 2: a clean, windy summer afternoon
Low particulates, high ozone, warm and breezy.

In [11]:
clean_day = {
    "PM10": 40.0, "SO2": 5.0, "NO2": 22.0, "CO": 500.0, "O3": 95.0,
    "TEMP": 27.0, "PRES": 1002.0, "DEWP": 14.0, "RAIN": 0.0, "WSPM": 3.2,
    "wd": "NW", "station": "Dingling", "hour": 15, "month": 7,
}
result = predict_air_quality(clean_day)
for k, v in result.items():
    print(f"{k:26s} {v}")

predicted_AQI_category     Moderate
unhealthy_probability      0.015
unhealthy_flag             False
predicted_PM25             26.9


### Example 3: batch prediction on genuinely unseen rows
Ten rows drawn from the held-out test set, compared against their true values.

In [12]:
batch = Xc_te.head(10).copy()
truth = yc_te.head(10)

out = pd.DataFrame({
    "Actual category":    truth.values,
    "Predicted category": loaded_multiclass.predict(batch),
    "P(unhealthy)":       loaded_binary.predict_proba(batch)[:, 1].round(3),
    "Predicted PM2.5":    loaded_regression.predict(batch).round(1),
})
out["Correct"] = np.where(out["Actual category"] == out["Predicted category"], "yes", "no")
out

,Actual category,Predicted category,P(unhealthy),Predicted PM2.5,Correct
0,Unhealthy,Unhealthy,0.903,97.4,yes
1,Unhealthy,Unhealthy,0.769,66.0,yes
2,Unhealthy,Unhealthy,0.576,68.5,yes
3,Unhealthy,Unhealthy,0.804,89.4,yes
4,Moderate,Moderate,0.010,21.7,yes
5,Unhealthy,Unhealthy,0.978,109.1,yes
6,Very Unhealthy,Very Unhealthy,0.995,200.4,yes
7,Unhealthy,Unhealthy,0.593,78.0,yes
8,Moderate,Good,0.000,11.4,no
9,Hazardous,Very Unhealthy,1.000,203.4,no


## 8. What Was Produced

| File | Contents |
|---|---|
| `models/model_aqi_multiclass.joblib` | Full pipeline: preprocessing + balanced Random Forest, 6-class AQI |
| `models/model_unhealthy_binary.joblib` | Full pipeline: preprocessing + tuned Random Forest, binary flag |
| `models/model_pm25_regression.joblib` | Full pipeline: preprocessing + Random Forest regressor |
| `models/model_metadata.joblib` | Feature order, class labels, threshold, provenance |

**Points established by this notebook:**

1. **The saved object is the whole pipeline, not just the estimator.** Preprocessing
   travels with the model, so a reloaded model accepts raw sensor input. This eliminates
   the most common deployment bug, where training and serving preprocess differently.
2. **Reload was verified, not assumed.** Section 6 asserts that predictions after
   reloading are identical, which is the actual test of whether persistence worked.
3. **The recommended threshold is stored with the model.** The 0.25 value derived in
   Notebook 03 is metadata, not folklore, a model shipped without its operating
   threshold is incomplete.
4. **The selected regression model is not the highest-scoring one**, and the reasoning
   is recorded. Deployment criteria legitimately include determinism, speed and
   interpretability alongside accuracy.

**Limitation to state honestly:** these models are trained on the same stratified
samples used throughout the project, not the full 420,768 rows. For a production system
one would retrain on all available data; for this assignment the sample keeps results
consistent with the reported figures and within the dataset size range the brief
specifies.
